In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings("ignore")

In [8]:
df = pd.read_csv("../data/parkinsons_tabular/parkinsons.csv").rename(columns={
    "MDVP:Fo(Hz)": "pitch", "HNR": "hnr",
    "MDVP:Jitter(Abs)": "jitter", "MDVP:Shimmer": "shimmer",
    "NHR": "nhr", "RPDE": "rpde", "DFA": "dfa"
})

# Extract subject ID from name column
# Names look like: phon_R01_S01_1 — subject is R01_S01
df["subject"] = df["name"].apply(lambda x: "_".join(x.split("_")[1:3]))

print(f"Total samples   : {len(df)}")
print(f"Unique subjects : {df['subject'].nunique()}")
print(f"Label dist      : {df['status'].value_counts().to_dict()}")
print(f"\nSamples per subject (first 10):")
print(df.groupby("subject")["status"].agg(["count","mean"]).head(10))

Total samples   : 195
Unique subjects : 32
Label dist      : {1: 147, 0: 48}

Samples per subject (first 10):
         count  mean
subject             
R01_S01      6   1.0
R01_S02      6   1.0
R01_S04      6   1.0
R01_S05      6   1.0
R01_S06      6   1.0
R01_S07      6   0.0
R01_S08      6   1.0
R01_S10      6   0.0
R01_S13      6   0.0
R01_S16      6   1.0


In [9]:
pk_cols = ["pitch", "hnr", "jitter", "shimmer", "nhr", "rpde", "dfa"]

subjects  = df["subject"].unique()
all_preds = []
all_probs = []
all_true  = []

print(f"Running Leave-One-Subject-Out validation...")
print(f"Total subjects: {len(subjects)}\n")

for subject in subjects:
    train_df = df[df["subject"] != subject]
    test_df  = df[df["subject"] == subject]

    X_train = train_df[pk_cols]
    y_train = train_df["status"]
    X_test  = test_df[pk_cols]
    y_test  = test_df["status"]

    imputer = SimpleImputer(strategy="mean")
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp  = imputer.transform(X_test)

    model = RandomForestClassifier(n_estimators=200, random_state=42)
    model.fit(X_train_imp, y_train)

    preds = model.predict(X_test_imp)
    probs = model.predict_proba(X_test_imp)[:, 1]

    all_preds.extend(preds)
    all_probs.extend(probs)
    all_true.extend(y_test)

all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_true  = np.array(all_true)

loso_acc = accuracy_score(all_true, all_preds)
loso_auc = roc_auc_score(all_true, all_probs)

print("="*60)
print("  Leave-One-Subject-Out (LOSO) Validation")
print("  Parkinson's Disease Detection")
print("="*60)
print(f"  Subjects evaluated : {len(subjects)}")
print(f"  Total predictions  : {len(all_preds)}")
print(f"  Accuracy           : {loso_acc:.4f}")
print(f"  AUROC              : {loso_auc:.4f}")
print("="*60)

Running Leave-One-Subject-Out validation...
Total subjects: 32

  Leave-One-Subject-Out (LOSO) Validation
  Parkinson's Disease Detection
  Subjects evaluated : 32
  Total predictions  : 195
  Accuracy           : 0.7128
  AUROC              : 0.6050


In [10]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

X_all = df[pk_cols]
y_all = df["status"]
imputer = SimpleImputer(strategy="mean")
X_imp = imputer.fit_transform(X_all)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model_cv = RandomForestClassifier(n_estimators=200, random_state=42)
cv_acc = cross_val_score(model_cv, X_imp, y_all, cv=skf, scoring="accuracy")
cv_auc = cross_val_score(model_cv, X_imp, y_all, cv=skf, scoring="roc_auc")

print("\nValidation Strategy Comparison:")
print("="*60)
print(f"  {'Strategy':<35} {'Accuracy':>10} {'AUROC':>10}")
print(f"  {'-'*55}")
print(f"  {'5-Fold Stratified CV':<35} {cv_acc.mean():>10.4f} {cv_auc.mean():>10.4f}")
print(f"  {'Leave-One-Subject-Out (LOSO)':<35} {loso_acc:>10.4f} {loso_auc:>10.4f}")
print(f"  {'-'*55}")
print(f"  {'Performance Difference':<35} "
      f"{loso_acc - cv_acc.mean():>+10.4f} "
      f"{loso_auc - cv_auc.mean():>+10.4f}")
print("="*60)
print()
print("  Note: LOSO is the gold standard for subject-independent")
print("  evaluation in clinical voice analysis. It ensures the")
print("  model is tested on entirely unseen subjects, preventing")
print("  any data leakage across recordings from the same person.")
print("="*60)


Validation Strategy Comparison:
  Strategy                              Accuracy      AUROC
  -------------------------------------------------------
  5-Fold Stratified CV                    0.8872     0.9494
  Leave-One-Subject-Out (LOSO)            0.7128     0.6050
  -------------------------------------------------------
  Performance Difference                 -0.1744    -0.3443

  Note: LOSO is the gold standard for subject-independent
  evaluation in clinical voice analysis. It ensures the
  model is tested on entirely unseen subjects, preventing
  any data leakage across recordings from the same person.


In [11]:
import os
os.makedirs("../results", exist_ok=True)

pd.DataFrame([{
    "cv_accuracy":   cv_acc.mean(),
    "cv_accuracy_std": cv_acc.std(),
    "cv_auroc":      cv_auc.mean(),
    "cv_auroc_std":  cv_auc.std(),
    "loso_accuracy": loso_acc,
    "loso_auroc":    loso_auc
}]).to_csv("../results/loso_validation.csv", index=False)
print("Saved: results/loso_validation.csv")

Saved: results/loso_validation.csv
